# [13.1] Diffusion and Image-Generation Controls

> **Local-first extension.** This notebook starts the image-generation interpretability track. The exercises use exact toy tensors to teach the control logic; the CUDA report checks pinned Stable Diffusion 1.5 safe-shape generations, CLIP alignment, DAAM-style attention localization, target-token ablation, image-quality controls, and white-noise rejection.

## Core Question

When an image-generation explanation shows a plausible heatmap or edited image, what would make the claim hard to fake?

The answer is not that the picture looks right. A useful first standard is: attention mass lands inside a named region, proposed denoising circuits beat random ablations, latent directions beat random directions, prompt-token ablations change the claimed region more than unrelated-token controls, generated images pass quality checks, and white-noise controls fail.

## Learning Objectives

By the end of this notebook you should be able to measure target-region attention, reject nonspecific denoising ablations, evaluate latent steering over random controls, test prompt-region causality, combine SD1.5 DAAM/token-ablation/quality/noise checks, and interpret a bounded CUDA report without overclaiming broad diffusion interpretability.


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t

GT_TIER = "GT-1"
EXERCISE_ID = "13_1_diffusion_and_image_generation_controls"
DIFFICULTY = 4
IMPORTANCE = 3
EXPECTED_RUNTIME = "seconds on toy contract; minutes on local real-model path"
REQUIRES_GPU = True

chapter = "chapter13_image_generation_interpretability"
section = "part1_diffusion_image_controls"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_diffusion_image_controls.tests as tests


In [ ]:
ImageDirection = Literal["increase", "decrease"]


@dataclass(frozen=True)
class AttentionRegionReport:
    region_mass: float
    off_region_mass: float
    region_selective: bool


@dataclass(frozen=True)
class DenoisingCircuitReport:
    baseline_loss: float
    ablated_loss: float
    random_control_loss: float
    ablation_delta: float
    random_delta: float
    circuit_specific: bool


@dataclass(frozen=True)
class LatentDirectionReport:
    baseline_mean: float
    steered_mean: float
    random_control_mean: float
    observed_delta: float
    random_delta: float
    has_directional_effect: bool


@dataclass(frozen=True)
class PromptRegionCausalReport:
    original_region_score: float
    ablated_region_score: float
    control_region_score: float
    target_drop: float
    control_drop: float
    prompt_region_causal: bool


@dataclass(frozen=True)
class DAAMRegionReport:
    target_region_mass: float
    control_region_mass: float
    mask_fraction: float
    captured_map_count: int
    target_control_gap: float
    target_lift_over_mask_fraction: float
    daam_localized: bool


@dataclass(frozen=True)
class TokenAblationReport:
    original_region_score: float
    target_ablated_region_score: float
    random_control_region_score: float
    target_drop: float
    random_control_drop: float
    target_ablation_passed: bool
    random_token_ablation_weaker: bool


@dataclass(frozen=True)
class ImageQualityReport:
    target_region_fraction: float
    rgb_std: float
    high_frequency_energy: float
    saturation_fraction: float
    image_quality_preserved: bool


@dataclass(frozen=True)
class WhiteNoiseImageReport:
    real_high_frequency_energy: float
    white_noise_high_frequency_energy: float
    white_noise_rejected: bool


@dataclass(frozen=True)
class SD15StrictReport:
    daam_passed: bool
    token_ablation_passed: bool
    random_token_ablation_weaker: bool
    image_quality_preserved: bool
    white_noise_rejected: bool
    sd15_strict_experiment_passed: bool


## Exercise 1 - Attention Region Maps

An attention map is not a result by itself. The result is a measured claim: this token puts enough mass inside this target region, and the rest is visible as off-region mass.

> Difficulty: easy  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_attention_region_report_measures_mass_and_rejects_bad_masks` passed!
```

The toy fixture has `region_mass = 0.6` and `off_region_mass = 0.4`.

</details>

<details>
<summary>Help - what should count as off-region mass?</summary>

Everything not selected by the target mask. Clamp negative attention to zero, renormalize by total mass, and reject an empty mask.

</details>

<details>
<summary>Common bugs</summary>

- Treating a heatmap screenshot as evidence without measuring mass.
- Forgetting to normalize after clamping negative values.
- Accepting an empty target mask.

</details>

<details>
<summary>Solution</summary>

Normalize the nonnegative map, sum over the boolean mask, and compare `region_mass` with `min_region_mass`.

</details>


In [ ]:
def attention_region_report(
    attention_map: t.Tensor,
    region_mask: t.Tensor,
    *,
    min_region_mass: float = 0.6,
) -> AttentionRegionReport:
    raise NotImplementedError()


tests.test_attention_region_report_measures_mass_and_rejects_bad_masks(
    attention_region_report
)


## Exercise 2 - Denoising Circuit Controls

A circuit ablation is only informative if it beats a same-size random-control ablation. Otherwise you have shown that damaging the model hurts, not that the proposed circuit is special.

> Difficulty: easy  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_denoising_circuit_report_requires_specificity` passed!
```

</details>

<details>
<summary>Help - what is the random-control gap?</summary>

Compute `ablation_delta = ablated_loss - baseline_loss` and `random_delta = random_control_loss - baseline_loss`. The proposed circuit should exceed both the minimum loss increase and the random-control delta by a margin.

</details>

<details>
<summary>Common bugs</summary>

- Checking only that the ablated loss is worse than baseline.
- Comparing raw losses instead of loss deltas from the same baseline.
- Forgetting that the random control can also hurt loss.

</details>

<details>
<summary>Solution</summary>

Return all three losses, both deltas, and set `circuit_specific` only when the target ablation clears both gates.

</details>


In [ ]:
def denoising_circuit_report(
    *,
    baseline_loss: float,
    ablated_loss: float,
    random_control_loss: float,
    min_loss_increase: float = 0.1,
    min_control_gap: float = 0.05,
) -> DenoisingCircuitReport:
    raise NotImplementedError()


tests.test_denoising_circuit_report_requires_specificity(denoising_circuit_report)


## Exercise 3 - Latent Direction Controls

A latent direction should change a score in the expected direction and beat random directions on the same paired examples.

> Difficulty: medium  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_latent_direction_report_requires_random_margin` passed!
```

</details>

<details>
<summary>Help - paired scores</summary>

Use means over paired baseline, steered, and random-control score tensors. Do not compare unrelated batches.

</details>

<details>
<summary>Common bugs</summary>

- Using absolute effect size but ignoring the requested direction.
- Letting a random direction pass when it explains almost the same effect.
- Accepting mismatched tensor shapes.

</details>

<details>
<summary>Solution</summary>

Compute `observed_delta` and `random_delta`, check the sign of the observed delta, then require an absolute margin over the random delta.

</details>


In [ ]:
def latent_direction_effect_report(
    baseline_scores: t.Tensor,
    steered_scores: t.Tensor,
    random_control_scores: t.Tensor,
    *,
    expected_direction: ImageDirection = "increase",
    min_effect: float = 0.2,
    min_random_margin: float = 0.1,
) -> LatentDirectionReport:
    raise NotImplementedError()


tests.test_latent_direction_report_requires_random_margin(latent_direction_effect_report)


## Exercise 4 - Prompt-Region Causality

If a token claims to control a visual region, ablating that token should change the target region more than ablating an unrelated control token.

> Difficulty: medium  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_prompt_region_report_requires_target_drop` passed!
```

</details>

<details>
<summary>Help - target drop versus control drop</summary>

Both drops are measured from the original region score. The target drop must clear an absolute threshold and exceed the control drop by a margin.

</details>

<details>
<summary>Common bugs</summary>

- Using `ablated - original` instead of `original - ablated`.
- Forgetting that unrelated-token ablations can also move the score.
- Treating a single edited image as causal evidence without a control edit.

</details>

<details>
<summary>Solution</summary>

Compute the target and control drops from the same original score, then gate on minimum target drop and control margin.

</details>


In [ ]:
def prompt_region_causal_report(
    *,
    original_region_score: float,
    ablated_region_score: float,
    control_region_score: float,
    min_target_drop: float = 0.2,
    min_control_margin: float = 0.1,
) -> PromptRegionCausalReport:
    raise NotImplementedError()


tests.test_prompt_region_report_requires_target_drop(prompt_region_causal_report)


## Exercise 5 - SD1.5 Control Reports

The real CUDA path uses pinned SD1.5 safe-shape generations, but the report logic should be understandable without loading a model. Here you implement the small report combinators used by the real-model path.

> Difficulty: medium-hard  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_sd15_toy_control_reports` passed!
```

</details>

<details>
<summary>Help - why so many reports?</summary>

Each report guards a different failure mode: plausible attention without localization, target-token edits that are no stronger than random edits, collapsed/blank images, and noisy images that trick superficial metrics.

</details>

<details>
<summary>Common bugs</summary>

- Counting target attention without comparing to a control token.
- Passing token ablation when the random/control ablation is just as strong.
- Ignoring white-noise controls after checking image quality.
- Combining reports with `any` instead of requiring every control to pass.

</details>

<details>
<summary>Solution</summary>

Implement each report as a small thresholded dataclass, then combine them with `all(...)` in `sd15_strict_acceptance_report`.

</details>


In [ ]:
def _as_hwc_rgb(image: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def color_region_mask(image: t.Tensor, target_color: Literal["red", "blue"]) -> t.Tensor:
    raise NotImplementedError()


def daam_region_report(
    *,
    target_region_mass: float,
    control_region_mass: float,
    mask_fraction: float,
    captured_map_count: int,
    min_target_control_gap: float = 0.005,
    min_lift_over_mask_fraction: float = 0.01,
    min_captured_map_count: int = 16,
) -> DAAMRegionReport:
    raise NotImplementedError()


def token_ablation_region_report(
    *,
    original_region_score: float,
    target_ablated_region_score: float,
    random_control_region_score: float,
    min_target_drop: float = 0.05,
    min_random_margin: float = 0.05,
) -> TokenAblationReport:
    raise NotImplementedError()


def image_quality_report(
    image: t.Tensor,
    *,
    target_color: Literal["red", "blue"],
    min_target_region_fraction: float = 0.02,
    min_rgb_std: float = 0.05,
    max_high_frequency_energy: float = 0.12,
) -> ImageQualityReport:
    raise NotImplementedError()


def white_noise_image_control_report(
    real_quality: ImageQualityReport,
    white_noise_image: t.Tensor,
    *,
    target_color: Literal["red", "blue"],
    max_high_frequency_energy: float = 0.12,
    min_noise_gap: float = 0.12,
) -> WhiteNoiseImageReport:
    raise NotImplementedError()


def sd15_strict_acceptance_report(
    *,
    daam_reports: tuple[DAAMRegionReport, ...] | list[DAAMRegionReport],
    token_ablation_reports: tuple[TokenAblationReport, ...] | list[TokenAblationReport],
    image_quality_reports: tuple[ImageQualityReport, ...] | list[ImageQualityReport],
    white_noise_reports: tuple[WhiteNoiseImageReport, ...] | list[WhiteNoiseImageReport],
) -> SD15StrictReport:
    raise NotImplementedError()


tests.test_sd15_toy_control_reports(
    daam_region_report,
    token_ablation_region_report,
    image_quality_report,
    white_noise_image_control_report,
    sd15_strict_acceptance_report,
)


## Exercise 6 - Notebook Contract and CUDA Report

The notebook contract has two surfaces. `run_smoke_test` executes the local toy controls. `run_gpu_test` reads the committed CUDA report produced by the full pinned SD1.5 path in `solutions.py` and checks the section-specific acceptance tests.

> Difficulty: medium  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
All tests in `test_committed_gpu_report_requires_sd15_strict_controls` passed!
```

</details>

<details>
<summary>Help - why not regenerate SD1.5 inside the learner notebook?</summary>

The full path loads pinned SD1.5, captures attention maps, runs target/control ablations, scores CLIP, and checks VRAM. That is the verification script's job. The notebook reads the committed report so students can inspect the real result without accidentally spending minutes or requiring model-cache access during every notebook execution.

</details>

<details>
<summary>Common bugs</summary>

- Returning only a boolean instead of the inspected metrics.
- Skipping the committed-report test.
- Confusing the toy contract with the real CUDA report.

</details>

<details>
<summary>Solution</summary>

`run_smoke_test` calls the local reports. `run_gpu_test` loads `verification_report.json`, passes it to the committed-report test, and returns `report["metrics"]["gpu_test"]`.

</details>


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    raise NotImplementedError()


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    raise NotImplementedError()


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    raise NotImplementedError()


tests.test_notebook_contract(run_smoke_test)
report_metrics = run_gpu_test()
committed_report = json.loads((section_dir / "verification_report.json").read_text())
tests.test_committed_gpu_report_requires_sd15_strict_controls(committed_report)
report_metrics


## Signature Result

A convincing 13.1 result is not a pretty generated image. It is a bounded report where the visualization and the causal controls agree.

| Check | Passing result |
|---|---:|
| Toy target-region attention mass | `0.60` |
| Toy denoising target vs random delta | `0.50` vs `0.15` |
| Toy latent target vs random delta | `0.60` vs `0.05` |
| SD1.5 CLIP retrieval | `1.0 / 1.0` |
| SD1.5 target-control attention gap | `>= 0.0147` |
| SD1.5 target-token ablation drop | `>= 0.1507` |
| SD1.5 white-noise high-frequency gap | `>= 0.2818` |
| Peak VRAM | `3.13 GB` |

<details>
<summary>Interpreting the result</summary>

The result supports the scoped claim: for pinned red-square and blue-circle SD1.5 prompts, target token attention localizes to the target region, target-token ablation causes the region to weaken more than the control ablation, and the generated images pass quality checks that white noise fails.

</details>

## Limitations

Supported: toy controls, pinned SD1.5 safe-shape CLIP/DAAM-style/token-ablation/quality/white-noise evidence, and report interpretation.

Not supported: broad diffusion interpretability, full DAAM replication, full denoising-circuit causal tracing, unsafe/NSFW prompts, diffusion LoRA controls, SDXL, video diffusion, or autoregressive image-token controls.

Deferred: denoising-step activation patching, diffusion LoRA features, AR image-token controls, multi-object circuits, and human perceptual evaluation.
